# Error Visualization and Analysis of Experimental Data Using Python
## Course: Data Exploration and Visualization (Mini Project)
---
**Author / GitHub:** [DineshMoorthy007](https://github.com/DineshMoorthy007)  
**Repository:** `error-visualization-experimental-data`  
**Technologies:** Python, Jupyter Notebook, Pandas, NumPy, Matplotlib, Seaborn, Git, GitHub

## 1. Aim
To perform systematic data exploration, quality verification, data preprocessing, outlier detection, experimental error analysis, and publication-quality visual diagnostics on measurement data from a simple pendulum experiment using Python, quantifying measurement uncertainties against theoretical physical predictions.

## 2. Dataset Description
The dataset comprises **80 experimental observations** of a simple pendulum oscillation across **8 distinct lengths** ($0.20\,\text{m}$ to $1.00\,\text{m}$), with **10 repeated measurement trials** per length configuration.

### Theoretical Physics Model
The ideal time period $T$ of a simple pendulum for small angular displacements is governed by:
$$
T = 2\pi \sqrt{\frac{L}{g}}
$$
where:
- $L$ = Pendulum length in metres ($\text{m}$)
- $T$ = Theoretical time period in seconds ($\text{s}$)
- $g$ = Acceleration due to gravity ($9.81\,\text{m/s}^2$)

### Dataset Attributes (Raw Data)
1. `Experiment_ID`: Unique alphanumeric trial identifier (`EXP001` to `EXP080`).
2. `Trial_Number`: Trial repetition index ($1$ to $10$).
3. `Length_m`: Pendulum length in metres ($0.20, 0.30, \dots, 1.00$).
4. `Theoretical_Period_s`: Mathematically calculated theoretical period in seconds.
5. `Experimental_Period_s`: Observed experimental time period in seconds.

## 3. Import Libraries
We import the foundational scientific, analytical, and visualization libraries required for exploratory analysis and plotting.

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display options and styling
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
sns.set_theme(style="whitegrid", font_scale=1.05)

print("Required libraries successfully imported!")

## 4. Load Dataset
We load the raw experimental dataset (`pendulum_experimental_data.csv`) using a relative path that works reliably across project directories.

In [ ]:
# Robust path resolution for loading raw dataset
raw_dataset_path = os.path.join('..', 'data', 'pendulum_experimental_data.csv')
if not os.path.exists(raw_dataset_path):
    raw_dataset_path = os.path.join('data', 'pendulum_experimental_data.csv')

df = pd.read_csv(raw_dataset_path)
print(f"Raw dataset successfully loaded from: {raw_dataset_path}")

## 5. Data Exploration
We perform foundational exploratory inspections to examine the dataset structure, head/tail records, dimensionality, and initial summary statistics.

In [ ]:
# A. First 5 records
print("=== First 5 Records (Head) ===")
df.head(5)

In [ ]:
# B. Last 5 records
print("=== Last 5 Records (Tail) ===")
df.tail(5)

In [ ]:
# C. Shape & D. Column Names
rows, cols = df.shape
print(f"Dataset Shape: {df.shape} (Rows: {rows}, Columns: {cols})")
print("\nAttribute Names:")
for i, col in enumerate(df.columns, start=1):
    print(f"  {i}. {col}")

In [ ]:
# E. Data Types & F. Dataset Information
print("=== Dataset Information (df.info()) ===")
df.info()

In [ ]:
# G. Descriptive Summary
print("=== Descriptive Statistical Summary (Raw Data) ===")
df.describe()

### Interpretation — Data Exploration
- **Dimensionality:** The raw dataset contains exactly **80 records** (rows) and **5 attributes** (columns).
- **Observations:** The dataset covers 8 distinct physical pendulum lengths ranging from $0.20\,\text{m}$ to $1.00\,\text{m}$, with 10 repeated trials per length.
- **Range Overview:** Theoretical periods range between $0.8971\,\text{s}$ and $2.0061\,\text{s}$, while experimental periods range from $0.8948\,\text{s}$ to $2.0485\,\text{s}$, confirming close alignment with theoretical physics expectations.

## 6. Data Quality Analysis
We conduct rigorous data quality verification, checking for missing values, duplicate observations, invalid data types, and physical measurement inconsistencies.

In [ ]:
# A. Missing Value Analysis
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100.0
missing_table = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Missing_Percentage (%)': missing_pct
})
print("=== Missing Value Analysis ===")
print(missing_table)
print(f"\nTotal Missing Values: {df.isnull().sum().sum()}")

In [ ]:
# B. Duplicate Record Check
duplicate_count = df.duplicated().sum()
print(f"Duplicate Records Count: {duplicate_count}")
if duplicate_count == 0:
    print("Validation Confirmation: No duplicate rows detected in dataset.")

In [ ]:
# C. Physical Validity Validation (Domain Integrity Constraints)
invalid_lengths = df[df['Length_m'] <= 0]
invalid_theo = df[df['Theoretical_Period_s'] <= 0]
invalid_exp = df[df['Experimental_Period_s'] <= 0]

print("=== Physical Constraint Validation ===")
print(f"1. Non-positive Pendulum Lengths (L <= 0)       : {len(invalid_lengths)} invalid records")
print(f"2. Non-positive Theoretical Periods (T_theo <= 0): {len(invalid_theo)} invalid records")
print(f"3. Non-positive Experimental Periods (T_exp <= 0) : {len(invalid_exp)} invalid records")

if len(invalid_lengths) == 0 and len(invalid_theo) == 0 and len(invalid_exp) == 0:
    print("\nValidation Result: All physical measurements are strictly positive and domain-valid.")

### Interpretation — Data Quality Analysis
- **Missing Values:** The dataset contains **0 missing/null values** across all attributes (100% complete).
- **Duplicates:** The dataset contains **0 duplicate records**, ensuring every trial is uniquely identified.
- **Physical Integrity:** All length values ($L > 0$) and time period values ($T > 0$) are strictly positive, satisfying physical boundary requirements for simple pendulum oscillations.

## 7. Data Preprocessing
We perform structured preprocessing: enforcing explicit data types, verifying column precision, and ensuring original measurement units (metres and seconds) remain unmodified for direct scientific interpretation.

In [ ]:
# Explicit type casting and standard precision enforcement
df_preprocessed = df.copy()
df_preprocessed['Experiment_ID'] = df_preprocessed['Experiment_ID'].astype(str)
df_preprocessed['Trial_Number'] = df_preprocessed['Trial_Number'].astype(int)
df_preprocessed['Length_m'] = df_preprocessed['Length_m'].astype(float).round(2)
df_preprocessed['Theoretical_Period_s'] = df_preprocessed['Theoretical_Period_s'].astype(float).round(4)
df_preprocessed['Experimental_Period_s'] = df_preprocessed['Experimental_Period_s'].astype(float).round(4)

print("Preprocessing complete. Data types and precision successfully standardized:")
print(df_preprocessed.dtypes)

### Interpretation — Data Preprocessing
- **Type Enforcement:** Verified that `Trial_Number` is integer, `Length_m` and period columns are float64, and `Experiment_ID` is string.
- **Precision Preservation:** Length measurements are formatted to 2 decimal places and period measurements to 4 decimal places.
- **Preservation of Units:** No artificial scaling or normalization was applied to maintain the physical meaning of measurements in metres (m) and seconds (s).

## 8. Variable Classification
We categorize all dataset variables into their statistical and scientific taxonomy with explicit academic rationales.

In [ ]:
variable_taxonomy = [
    {
        'Variable Name': 'Experiment_ID',
        'Data Type': 'Object (String)',
        'Variable Class': 'Identifier (Nominal)',
        'Role in Experiment': 'Unique key identifying individual observation records (EXP001 to EXP080).'
    },
    {
        'Variable Name': 'Trial_Number',
        'Data Type': 'Integer',
        'Variable Class': 'Numerical (Discrete)',
        'Role in Experiment': 'Repetition index (1 to 10) for trials at a given pendulum length.'
    },
    {
        'Variable Name': 'Length_m',
        'Data Type': 'Float',
        'Variable Class': 'Numerical (Continuous)',
        'Role in Experiment': 'Independent physical variable: length of the pendulum in metres.'
    },
    {
        'Variable Name': 'Theoretical_Period_s',
        'Data Type': 'Float',
        'Variable Class': 'Numerical (Continuous)',
        'Role in Experiment': 'Reference value: ideal oscillation period from T = 2*pi*sqrt(L/g).'
    },
    {
        'Variable Name': 'Experimental_Period_s',
        'Data Type': 'Float',
        'Variable Class': 'Numerical (Continuous)',
        'Role in Experiment': 'Dependent measured variable: observed time period in seconds.'
    }
]

taxonomy_df = pd.DataFrame(variable_taxonomy)
print("=== Variable Classification Table ===")
taxonomy_df

### Interpretation — Variable Classification
- `Length_m` serves as the controlled **independent variable**.
- `Experimental_Period_s` is the **dependent empirical response variable**.
- `Theoretical_Period_s` acts as the **analytical reference standard**.
- `Trial_Number` captures repeated experimental trials under identical controlled conditions.

## 9. Error Calculation
We compute four core experimental discrepancy metrics comparing experimental observations against theoretical reference values:

1. **Error ($\text{Error}_s$):** $\text{Error} = T_{\text{exp}} - T_{\text{theo}}$ (signed deviation)
2. **Absolute Error ($\text{Absolute\_Error}_s$):** $|T_{\text{exp}} - T_{\text{theo}}|$ (error magnitude in seconds)
3. **Relative Error ($\text{Relative\_Error}$):** $\frac{|T_{\text{exp}} - T_{\text{theo}}|}{T_{\text{theo}}}$ (dimensionless ratio)
4. **Percentage Error ($\text{Percentage\_Error}$):** $\text{Relative\_Error} \times 100\%$ (normalized percentage error)

In [ ]:
# Compute experimental error metrics
df_preprocessed['Error_s'] = (
    df_preprocessed['Experimental_Period_s'] - df_preprocessed['Theoretical_Period_s']
).round(4)

df_preprocessed['Absolute_Error_s'] = (
    df_preprocessed['Error_s'].abs()
).round(4)

df_preprocessed['Relative_Error'] = (
    df_preprocessed['Absolute_Error_s'] / df_preprocessed['Theoretical_Period_s']
).round(6)

df_preprocessed['Percentage_Error'] = (
    df_preprocessed['Relative_Error'] * 100.0
).round(4)

print("=== Error Calculation Sample (First 10 Records) ===")
df_preprocessed[['Experiment_ID', 'Length_m', 'Theoretical_Period_s', 'Experimental_Period_s', 'Error_s', 'Absolute_Error_s', 'Percentage_Error']].head(10)

### Interpretation — Error Calculation
- **Signed Error:** Both positive and negative discrepancies occur, representing slight stopwatch trigger timing variances.
- **Magnitude:** The mean absolute error is approximately **$0.0214\,\text{s}$**, which is consistent with typical human reaction time in manual laboratory timing.
- **Relative Precision:** The mean percentage error across all 80 trials is **$1.52\%$**, reflecting high overall measurement fidelity.

## 10. Outlier Detection
We apply the standard **Interquartile Range (IQR)** method to detect statistical anomalies in the experimental error distribution.

### IQR Formulation
- $Q_1 = \text{25th percentile}$
- $Q_3 = \text{75th percentile}$
- $\text{IQR} = Q_3 - Q_1$
- $\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}$
- $\text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$

*Note: Outliers in experimental data represent important physical variations (such as timing perturbations or reaction latency) and are flagged with `Outlier_Flag` rather than deleted.*

In [ ]:
# Outlier detection on Percentage_Error using IQR method
q1 = df_preprocessed['Percentage_Error'].quantile(0.25)
q3 = df_preprocessed['Percentage_Error'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Flag outliers
df_preprocessed['Outlier_Flag'] = (
    (df_preprocessed['Percentage_Error'] < lower_bound) | (df_preprocessed['Percentage_Error'] > upper_bound)
)

outliers_df = df_preprocessed[df_preprocessed['Outlier_Flag']]

print("=== IQR Outlier Analysis Summary ===")
print(f"First Quartile (Q1)     : {q1:.4f} %")
print(f"Third Quartile (Q3)     : {q3:.4f} %")
print(f"Interquartile Range(IQR): {iqr:.4f} %")
print(f"Lower Outlier Threshold : {lower_bound:.4f} %")
print(f"Upper Outlier Threshold : {upper_bound:.4f} %")
print(f"Potential Outliers Count: {len(outliers_df)} out of {len(df_preprocessed)} records ({len(outliers_df)/len(df_preprocessed)*100:.2f}%)")
print(f"Outlier Experiment IDs  : {outliers_df['Experiment_ID'].tolist()}")

print("\n=== Outlier Records Detail ===")
outliers_df[['Experiment_ID', 'Length_m', 'Theoretical_Period_s', 'Experimental_Period_s', 'Percentage_Error', 'Outlier_Flag']]

### Interpretation — Outlier Detection
- **Identified Outliers:** Exactly **5 potential outliers** were detected based on the $1.5 \times \text{IQR}$ threshold on percentage error (`EXP004`, `EXP007`, `EXP014`, `EXP038`, `EXP055`).
- **Diagnostic Value:** These observations correspond to trials with higher reaction delays or stopwatch pressing offsets ($> 4.00\%$ error), providing valuable test cases for residual and dispersion analysis.
- **Retention Policy:** Outliers are retained in the dataset with `Outlier_Flag = True` to preserve empirical transparency and prevent data distortion.

## 11. Error Classification
To support qualitative error analysis and future categorical visualizations, we categorize trials into project-defined analytical error quality tiers:
- **Low Error (< 1%):** High-precision measurements with minimal timing error.
- **Moderate Error (1% to 2%):** Acceptable laboratory measurement variance.
- **High Error (>= 2%):** Noticeable timing offsets or experimental perturbations.

In [ ]:
# Discretize into error quality categories
conditions = [
    df_preprocessed['Percentage_Error'] < 1.0,
    (df_preprocessed['Percentage_Error'] >= 1.0) & (df_preprocessed['Percentage_Error'] < 2.0),
    df_preprocessed['Percentage_Error'] >= 2.0
]
categories = ['Low Error (<1%)', 'Moderate Error (1-2%)', 'High Error (>=2%)']

df_preprocessed['Error_Category'] = np.select(conditions, categories, default='Unclassified')

# Summary distribution
category_counts = df_preprocessed['Error_Category'].value_counts().reindex(categories, fill_value=0)
category_pct = (category_counts / len(df_preprocessed)) * 100.0

category_summary_df = pd.DataFrame({
    'Error_Category': category_counts.index,
    'Observation_Count': category_counts.values,
    'Percentage_Share (%)': category_pct.values
})

print("=== Error Category Distribution ===")
category_summary_df

### Interpretation — Error Classification
- **Low Error (< 1%):** Accounts for **39 observations (48.75%)** of the dataset, demonstrating high experimental repeatability.
- **Moderate Error (1% to 2%):** Accounts for **23 observations (28.75%)**.
- **High Error (>= 2%):** Accounts for **18 observations (22.50%)**, which includes the 5 IQR-flagged outliers.
- **Cumulative Performance:** **77.50%** of all observations exhibit measurement errors strictly under $2.0\%$.

## 12. Descriptive Statistical Analysis
We calculate the full suite of descriptive statistical metrics required for the college mini project lab record across key experimental variables.

In [ ]:
# Comprehensive descriptive statistical calculations
target_variables = ['Experimental_Period_s', 'Absolute_Error_s', 'Percentage_Error']
stats_rows = []

for col in target_variables:
    s = df_preprocessed[col]
    mean_val = s.mean()
    median_val = s.median()
    min_val = s.min()
    max_val = s.max()
    range_val = max_val - min_val
    var_val = s.var()
    std_val = s.std()
    q1_val = s.quantile(0.25)
    q2_val = s.quantile(0.50)
    q3_val = s.quantile(0.75)
    
    # Continuous mode note
    rounded_mode = s.round(2).mode()
    mode_str = f"{rounded_mode.iloc[0]:.2f} (rounded)" if not rounded_mode.empty else "N/A"
    
    stats_rows.append({
        'Metric_Variable': col,
        'Mean': mean_val,
        'Median (Q2)': median_val,
        'Mode': mode_str,
        'Minimum': min_val,
        'Maximum': max_val,
        'Range': range_val,
        'Variance': var_val,
        'Std_Deviation': std_val,
        'Q1 (25th %)': q1_val,
        'Q2 (50th %)': q2_val,
        'Q3 (75th %)': q3_val
    })

stats_summary_table = pd.DataFrame(stats_rows)
print("=== Comprehensive Descriptive Statistics Table ===")
stats_summary_table

### Interpretation — Descriptive Statistics
- **Central Tendency:** Mean experimental period is **$1.4704\,\text{s}$** with a median of **$1.4952\,\text{s}$**, matching closely across the multi-length distribution.
- **Error Dispersion:** The mean absolute error is **$0.0214\,\text{s}$** with a low standard deviation of **$0.0267\,\text{s}$**, confirming tight experimental clustering.
- **Percentage Error Distribution:** The median percentage error is **$1.0119\%$**, while the 75th percentile ($Q_3$) is **$1.8374\%$**, confirming that half of all measurements have under $\approx 1\%$ error.

## 13. Length-wise Summary
We group observations by pendulum length (`Length_m`) to evaluate experimental precision, variability, and maximum percentage error across different physical configurations.

In [ ]:
# Grouped summary by pendulum length
length_summary = df_preprocessed.groupby('Length_m').agg(
    Observation_Count=('Experiment_ID', 'count'),
    Mean_Exp_Period_s=('Experimental_Period_s', 'mean'),
    Mean_Theo_Period_s=('Theoretical_Period_s', 'first'),
    Mean_Absolute_Error_s=('Absolute_Error_s', 'mean'),
    Mean_Percentage_Error=('Percentage_Error', 'mean'),
    Std_Exp_Period_s=('Experimental_Period_s', 'std'),
    Max_Percentage_Error=('Percentage_Error', 'max')
).reset_index()

print("=== Length-wise Aggregation Summary ===")
length_summary

### Interpretation — Length-wise Summary
- **Length Scaling:** Mean experimental period steadily increases from **$0.9150\,\text{s}$** ($L = 0.20\,\text{m}$) to **$2.0012\,\text{s}$** ($L = 1.00\,\text{m}$), demonstrating non-linear square-root scaling ($T \propto \sqrt{L}$).
- **Standard Deviation Stability:** Across all 8 length groups, the standard deviation of experimental period remains consistently small (between $0.0159\,\text{s}$ and $0.0533\,\text{s}$).
- **Relative Error Stability:** Mean percentage error per length group stays reliably between $0.91\%$ and $2.10\%$, confirming stable experimental precision across all pendulum dimensions.

## 14. Data Visualization
In this section, we implement the complete suite of **seven publication-quality visualizations** required by the Data Exploration and Visualization course template, covering all major chart types: Line Chart, Bar Chart, Pie Chart, Box Plot, Scatter Plot, Histogram, and Correlation Heatmap.

### 14.1 Experimental vs Theoretical Period — Line Chart
**Purpose:** Compares empirical mean experimental periods against theoretical physics predictions across all physical pendulum lengths.

In [ ]:
# 14.1 Line Chart: Experimental vs Theoretical Period
fig, ax = plt.subplots(figsize=(9, 5.5))

grouped_line = df_preprocessed.groupby('Length_m').agg(
    Mean_Exp_Period=('Experimental_Period_s', 'mean'),
    Std_Exp_Period=('Experimental_Period_s', 'std'),
    Theoretical_Period=('Theoretical_Period_s', 'first')
).reset_index().sort_values('Length_m')

lengths_dense = np.linspace(0.15, 1.05, 200)
theoretical_dense = 2.0 * np.pi * np.sqrt(lengths_dense / 9.81)

ax.plot(
    lengths_dense,
    theoretical_dense,
    color='#1f77b4',
    linestyle='--',
    linewidth=2.2,
    label=r'Theoretical Model: $T = 2\pi\sqrt{L/g}$ ($g=9.81\,\mathrm{m/s^2}$)'
)

ax.errorbar(
    grouped_line['Length_m'],
    grouped_line['Mean_Exp_Period'],
    yerr=grouped_line['Std_Exp_Period'],
    fmt='o-',
    color='#d62728',
    ecolor='#7f7f7f',
    elinewidth=1.2,
    capsize=4,
    capthick=1.2,
    markersize=6.5,
    linewidth=1.8,
    label=r'Mean Experimental Period $\pm 1\,\mathrm{SD}$'
)

ax.set_title('Experimental vs Theoretical Pendulum Period', pad=12, fontweight='bold')
ax.set_xlabel('Pendulum Length (m)', fontweight='semibold')
ax.set_ylabel('Time Period (s)', fontweight='semibold')
ax.set_xlim(0.15, 1.05)
ax.set_ylim(0.75, 2.15)
ax.legend(loc='upper left', frameon=True, framealpha=0.9)
plt.tight_layout()

# Save figure
os.makedirs('../visualizations', exist_ok=True)
plt.savefig('../visualizations/01_experimental_vs_theoretical_line.png', dpi=300)
plt.show()

**Interpretation (14.1 Line Chart):**
- The mean experimental time period closely traces the theoretical curve $T = 2\pi\sqrt{L/g}$ across all 8 tested pendulum lengths ($0.20\,\text{m}$ to $1.00\,\text{m}$).
- The small standard deviation error bars (ranging from $\pm 0.0159\,\text{s}$ at $0.20\,\text{m}$ to $\pm 0.0533\,\text{s}$ at $0.70\,\text{m}$) demonstrate high empirical measurement repeatability under laboratory conditions.
- The non-linear square-root growth trend demonstrates that doubling the pendulum length does not double the period, verifying the theoretical power-law relationship $T \propto L^{0.5}$.

### 14.2 Average Percentage Error by Length — Bar Chart
**Purpose:** Examines how the average measurement error varies across different pendulum lengths.

In [ ]:
# 14.2 Bar Chart: Average Percentage Error by Length
fig, ax = plt.subplots(figsize=(9, 5.5))

grouped_bar = df_preprocessed.groupby('Length_m')['Percentage_Error'].mean().reset_index().sort_values('Length_m')
x_labels = [f"{length:.2f} m" for length in grouped_bar['Length_m']]

bars = ax.bar(
    x_labels,
    grouped_bar['Percentage_Error'],
    color='#2b5c8f',
    edgecolor='#1a365d',
    width=0.55,
    alpha=0.88
)

for bar in bars:
    height = bar.get_height()
    ax.annotate(
        f"{height:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 4),
        textcoords="offset points",
        ha='center',
        va='bottom',
        fontsize=9.5,
        fontweight='bold',
        color='#1a365d'
    )

overall_mean = df_preprocessed['Percentage_Error'].mean()
ax.axhline(
    overall_mean,
    color='#d9534f',
    linestyle='--',
    linewidth=1.5,
    label=f'Dataset Overall Mean ({overall_mean:.2f}%)'
)

ax.set_title('Average Percentage Error by Pendulum Length', pad=12, fontweight='bold')
ax.set_xlabel('Pendulum Length (m)', fontweight='semibold')
ax.set_ylabel('Mean Percentage Error (%)', fontweight='semibold')
ax.set_ylim(0, max(grouped_bar['Percentage_Error']) * 1.25)
ax.legend(loc='upper right', frameon=True)
plt.tight_layout()

# Save figure
plt.savefig('../visualizations/02_percentage_error_bar.png', dpi=300)
plt.show()

**Interpretation (14.2 Bar Chart):**
- The highest mean percentage error occurs at $L = 0.50\,\text{m}$ ($2.10\%$) and $L = 0.20\,\text{m}$ ($2.05\%$), while the lowest average percentage error is observed at $L = 0.80\,\text{m}$ ($0.91\%$) and $L = 0.60\,\text{m}$ ($1.00\%$).
- Across the 8 physical lengths, the average percentage error exhibits an irregular pattern rather than a monotonic increase or decrease, indicating that human reaction timing noise is stochastically distributed across length configurations.
- All length-averaged percentage errors remain within a narrow band between $0.91\%$ and $2.10\%$, closely aligned around the dataset overall mean of $1.52\%$.

### 14.3 Error Category Distribution — Pie Chart
**Purpose:** Visualizes the proportional composition of experimental observations across project-defined analytical error quality tiers.

In [ ]:
# 14.3 Pie Chart: Error Category Distribution
fig, ax = plt.subplots(figsize=(8, 6))

categories = ['Low Error (<1%)', 'Moderate Error (1-2%)', 'High Error (>=2%)']
counts = df_preprocessed['Error_Category'].value_counts().reindex(categories, fill_value=0)
colors = ['#2ca02c', '#ff7f0e', '#d62728']
explode = (0.04, 0.02, 0.06)

def format_autopct(pct):
    total = sum(counts)
    val = int(round(pct * total / 100.0))
    return f"{pct:.1f}%\n({val} obs)"

wedges, texts, autotexts = ax.pie(
    counts,
    labels=categories,
    autopct=format_autopct,
    startangle=140,
    colors=colors,
    explode=explode,
    textprops={'fontsize': 10, 'fontweight': 'medium'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5, 'antialiased': True}
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(9.5)
    autotext.set_weight('bold')

ax.set_title('Distribution of Experimental Error Categories', pad=14, fontweight='bold')
plt.tight_layout()

# Save figure
plt.savefig('../visualizations/03_error_category_pie.png', dpi=300)
plt.show()

**Interpretation (14.3 Pie Chart):**
- The **Low Error (< 1%)** category comprises the largest proportion of observations, representing **48.8% (39 observations)** of the entire experimental dataset.
- The **Moderate Error (1% to 2%)** tier accounts for **28.8% (23 observations)**, while the **High Error (>= 2%)** tier constitutes **22.5% (18 observations)**.
- In aggregate, **77.5% of all measurements** fall within $< 2.0\%$ deviation from theoretical physics predictions, confirming high experimental precision with only a small minority of perturbed trials.

### 14.4 Experimental Period Distribution — Box Plot
**Purpose:** Displays the median, interquartile spread, dispersion, and potential outliers of observed experimental periods across length groups.

In [ ]:
# 14.4 Box Plot: Experimental Period Distribution by Length
fig, ax = plt.subplots(figsize=(10, 6))

lengths_sorted = sorted(df_preprocessed['Length_m'].unique())
data_by_length = [df_preprocessed[df_preprocessed['Length_m'] == l]['Experimental_Period_s'].values for l in lengths_sorted]
theo_by_length = [df_preprocessed[df_preprocessed['Length_m'] == l]['Theoretical_Period_s'].iloc[0] for l in lengths_sorted]

box = ax.boxplot(
    data_by_length,
    tick_labels=[f"{l:.2f} m" for l in lengths_sorted],
    patch_artist=True,
    showmeans=True,
    meanline=True,
    flierprops=dict(marker='o', markerfacecolor='#d62728', markersize=6.5, linestyle='none', markeredgecolor='black'),
    medianprops=dict(color='#1a365d', linewidth=2.0),
    meanprops=dict(color='#2ca02c', linestyle='--', linewidth=1.5),
    boxprops=dict(facecolor='#dbeafe', color='#1e3a8a', linewidth=1.2),
    whiskerprops=dict(color='#1e3a8a', linewidth=1.2),
    capprops=dict(color='#1e3a8a', linewidth=1.2)
)

ax.scatter(
    range(1, len(lengths_sorted) + 1),
    theo_by_length,
    color='#b91c1c',
    marker='D',
    s=45,
    zorder=5,
    label=r'Theoretical Reference $T_{\mathrm{theo}}$'
)

ax.plot([], [], color='#1a365d', linewidth=2, label='Median Period')
ax.plot([], [], color='#2ca02c', linestyle='--', linewidth=1.5, label='Mean Period')
ax.scatter([], [], marker='o', color='#d62728', s=40, label='Statistical Outlier Point')

ax.set_title('Distribution of Experimental Period by Pendulum Length', pad=12, fontweight='bold')
ax.set_xlabel('Pendulum Length (m)', fontweight='semibold')
ax.set_ylabel('Experimental Period (s)', fontweight='semibold')
ax.legend(loc='upper left', frameon=True, framealpha=0.9)
plt.tight_layout()

# Save figure
plt.savefig('../visualizations/04_experimental_period_boxplot.png', dpi=300)
plt.show()

**Interpretation (14.4 Box Plot):**
- The median experimental period increases monotonically from $0.9166\,\text{s}$ ($L = 0.20\,\text{m}$) to $2.0116\,\text{s}$ ($L = 1.00\,\text{m}$), aligning closely with theoretical diamond markers at each length.
- Visible statistical outlier trials are present in the box distributions for lengths $0.30\,\text{m}$, $0.40\,\text{m}$, $0.50\,\text{m}$, and $0.70\,\text{m}$, corresponding to the higher-error trials identified during IQR analysis.
- The interquartile range (IQR) box heights remain consistently tight across all groups, with the greatest spread occurring at $L = 0.70\,\text{m}$ ($\text{SD} = 0.0533\,\text{s}$) due to trial `EXP055` ($T_{\text{exp}} = 1.8506\,\text{s}$).

### 14.5 Length vs Experimental Period — Scatter Plot
**Purpose:** Examines the point-by-point scatter distribution of all 80 individual experimental trials against the theoretical physics reference curve.

In [ ]:
# 14.5 Scatter Plot: Pendulum Length vs Experimental Period
fig, ax = plt.subplots(figsize=(9, 5.5))

inliers = df_preprocessed[~df_preprocessed['Outlier_Flag']]
outliers = df_preprocessed[df_preprocessed['Outlier_Flag']]

ax.scatter(
    inliers['Length_m'],
    inliers['Experimental_Period_s'],
    color='#2563eb',
    alpha=0.75,
    edgecolors='#1e3a8a',
    linewidth=0.8,
    s=55,
    label=f'Experimental Trials (Inliers, n={len(inliers)})'
)

if len(outliers) > 0:
    ax.scatter(
        outliers['Length_m'],
        outliers['Experimental_Period_s'],
        color='#dc2626',
        alpha=0.95,
        edgecolors='#7f1d1d',
        linewidth=1.2,
        s=75,
        marker='^',
        label=f'Flagged Outlier Trials (n={len(outliers)})'
    )

lengths_dense = np.linspace(0.15, 1.05, 200)
theoretical_dense = 2.0 * np.pi * np.sqrt(lengths_dense / 9.81)
ax.plot(
    lengths_dense,
    theoretical_dense,
    color='#0f172a',
    linestyle='--',
    linewidth=2.0,
    label=r'Theoretical Reference: $T = 2\pi\sqrt{L/g}$'
)

ax.set_title('Pendulum Length vs Experimental Period', pad=12, fontweight='bold')
ax.set_xlabel('Pendulum Length (m)', fontweight='semibold')
ax.set_ylabel('Experimental Period (s)', fontweight='semibold')
ax.set_xlim(0.15, 1.05)
ax.set_ylim(0.75, 2.15)
ax.legend(loc='upper left', frameon=True, framealpha=0.9)
plt.tight_layout()

# Save figure
plt.savefig('../visualizations/05_length_vs_period_scatter.png', dpi=300)
plt.show()

**Interpretation (14.5 Scatter Plot):**
- A strong positive non-linear relationship exists between pendulum length and experimental oscillation period, with all 80 individual trials closely following the theoretical trajectory $T = 2\pi\sqrt{L/g}$.
- The 75 inlier observations form tight clusters along the curve, while the 5 highlighted outlier trials ($n=5$) exhibit distinct vertical offsets reflecting momentary stopwatch reaction delays.
- There are no severe distortions or physically implausible outliers, confirming that the experimental data follows consistent gravitational dynamics.

### 14.6 Percentage Error Distribution — Histogram
**Purpose:** Evaluates the skewness, spread, and modal concentration of percentage error across observations.

In [ ]:
# 14.6 Histogram: Distribution of Percentage Error
fig, ax = plt.subplots(figsize=(9, 5.5))

mean_err = df_preprocessed['Percentage_Error'].mean()
median_err = df_preprocessed['Percentage_Error'].median()

sns.histplot(
    df_preprocessed['Percentage_Error'],
    bins=16,
    kde=True,
    color='#3b82f6',
    edgecolor='#1e3a8a',
    alpha=0.65,
    ax=ax
)

ax.axvline(
    mean_err,
    color='#dc2626',
    linestyle='--',
    linewidth=2.0,
    label=f'Mean Percentage Error ({mean_err:.2f}%)'
)
ax.axvline(
    median_err,
    color='#16a34a',
    linestyle='-.',
    linewidth=2.0,
    label=f'Median Percentage Error ({median_err:.2f}%)'
)

ax.set_title('Distribution of Percentage Error', pad=12, fontweight='bold')
ax.set_xlabel('Percentage Error (%)', fontweight='semibold')
ax.set_ylabel('Frequency (Number of Observations)', fontweight='semibold')
ax.legend(loc='upper right', frameon=True, framealpha=0.9)
plt.tight_layout()

# Save figure
plt.savefig('../visualizations/06_error_distribution_histogram.png', dpi=300)
plt.show()

**Interpretation (14.6 Histogram):**
- The percentage error distribution is strongly right-skewed (positively skewed), with the vast majority of observations concentrated in the low-error interval ($0.0\%$ to $2.0\%$).
- The median percentage error ($1.01\%$) is notably lower than the mean percentage error ($1.52\%$), reflecting the pulling effect of a small number of right-tail outlier observations ($> 4.00\%$, reaching up to $10.29\%$).
- Over 60% of all observations record less than $1.5\%$ error, demonstrating that large measurement discrepancies are rare and isolated occurrences.

### 14.7 Correlation Heatmap of Experimental Variables
**Purpose:** Analyzes bivariate linear dependencies across physical dimensions and error metrics using Pearson correlation coefficients.

In [ ]:
# 14.7 Heatmap: Correlation Heatmap of Experimental Variables
fig, ax = plt.subplots(figsize=(8.5, 7))

numeric_cols = [
    'Length_m',
    'Theoretical_Period_s',
    'Experimental_Period_s',
    'Error_s',
    'Absolute_Error_s',
    'Relative_Error',
    'Percentage_Error'
]

available_cols = [c for c in numeric_cols if c in df_preprocessed.columns]
corr_matrix = df_preprocessed[available_cols].corr()

clean_labels = [
    'Length (m)',
    'Theo Period (s)',
    'Exp Period (s)',
    'Signed Error (s)',
    'Abs Error (s)',
    'Relative Error',
    '% Error (%)'
]

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='vlag',
    vmin=-1.0,
    vmax=1.0,
    square=True,
    linewidths=1.0,
    cbar_kws={'label': 'Pearson Correlation Coefficient ($r$)', 'shrink': 0.82},
    xticklabels=clean_labels[:len(available_cols)],
    yticklabels=clean_labels[:len(available_cols)],
    ax=ax
)

ax.set_title('Correlation Heatmap of Experimental Variables', pad=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# Save figure
plt.savefig('../visualizations/07_correlation_heatmap.png', dpi=300)
plt.show()

**Interpretation (14.7 Correlation Heatmap):**
- Pendulum length (`Length_m`), theoretical period (`Theoretical_Period_s`), and experimental period (`Experimental_Period_s`) exhibit near-perfect positive Pearson correlations ($r \approx 0.99 - 1.00$), validating the underlying physical law.
- Error metrics (`Absolute_Error_s`, `Relative_Error`, and `Percentage_Error`) show extremely strong mutual correlations ($r \ge 0.96$), as relative and percentage errors scale directly with absolute deviation.
- Crucially, length shows negligible correlation with signed error ($r = -0.06$) and percentage error ($r = -0.12$), indicating that experimental percentage error is independent of pendulum length and that timing inaccuracies were not systematically biased by length (noting that correlation does not imply causation).